In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_validate
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.base import clone

import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# STAGE 6 CONFIGURATION
# ============================================================

# Change these paths according to your project folder
SF24_PATH = "data/SF24.csv"
AGRIYIELD_PATH = "data/AgriYield_2025.csv"

# IMPORTANT:
# Replace these with the ACTUAL target column names
SF24_TARGET = "crop"
AGRIYIELD_TARGET = "yield"

# Test size
TEST_SIZE = 0.20

# Reproducibility
RANDOM_STATE = 42

# Number of CV folds
N_SPLITS = 5

print("Configuration loaded.")
print("Random state:", RANDOM_STATE)
print("Test size:", TEST_SIZE)
print("CV folds:", N_SPLITS)

Configuration loaded.
Random state: 42
Test size: 0.2
CV folds: 5


In [5]:
import sqlite3
# ============================================================
# STAGE 6.2 — LOAD DATA FROM SQL DATABASE
# ============================================================
db_path = r"C:\Users\ADMIN\smart agricultural analytics\agriculture.db"
conn = sqlite3.connect(db_path)

# Check available tables
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)

print("Available tables:")
display(tables)

Available tables:


,name
0,crop_recommendation
1,agri_yield_train
2,agri_yield_test
3,crop_recommendation_clean
4,agri_yield_train_clean
5,agri_yield_test_clean
6,crop_recommendation_fe
7,agri_yield_train_fe
8,agri_yield_test_fe
9,crop_recommendation_selected


In [7]:
# Load final selected datasets
sf24 = pd.read_sql_query(
    "SELECT * FROM crop_recommendation_selected",
    conn
)

agriyield_train = pd.read_sql_query(
    "SELECT * FROM agri_yield_train_selected",
    conn
)

agriyield_test = pd.read_sql_query(
    "SELECT * FROM agri_yield_test_selected",
    conn
)

print("Tables loaded successfully.")

print("\nSF24 / Crop Recommendation:")
print(sf24.shape)

print("\nAgriYield Train:")
print(agriyield_train.shape)

print("\nAgriYield Test:")
print(agriyield_test.shape)

Tables loaded successfully.

SF24 / Crop Recommendation:
(2200, 13)

AgriYield Train:
(10000, 11)

AgriYield Test:
(2000, 10)


In [8]:
# ============================================================
# STAGE 6.3 — INSPECT LOADED TABLES
# ============================================================

print("=" * 60)
print("CROP RECOMMENDATION SELECTED")
print("=" * 60)

display(sf24.head())

print("\nColumns:")
print(sf24.columns.tolist())


print("\n" + "=" * 60)
print("AGRI YIELD TRAIN SELECTED")
print("=" * 60)

display(agriyield_train.head())

print("\nColumns:")
print(agriyield_train.columns.tolist())


print("\n" + "=" * 60)
print("AGRI YIELD TEST SELECTED")
print("=" * 60)

display(agriyield_test.head())

print("\nColumns:")
print(agriyield_test.columns.tolist())

CROP RECOMMENDATION SELECTED


,humidity,K,temperature,N,P_K_Ratio,NPK_Total,Temperature_Rainfall,rainfall,N_K_Ratio,Humidity_Rainfall,Temperature_Humidity,P,label
0,82.002744,43,20.879744,90,0.976744,175,4237.241986,202.935536,2.093023,16641.270870,1712.196283,42,rice
1,80.319644,41,21.770462,85,1.414634,184,4934.395694,226.655537,2.073171,18204.892093,1748.595734,58,rice
2,82.320763,44,23.004459,60,1.250000,159,6072.354751,263.964248,1.363636,21729.738241,1893.744627,55,rice
3,80.158363,40,26.491096,74,0.875000,149,6433.734530,242.864034,1.850000,19467.583326,2123.482908,35,rice
4,81.604873,42,20.130175,78,1.000000,162,5288.545993,262.717340,1.857143,21439.015172,1642.720357,42,rice



Columns:
['humidity', 'K', 'temperature', 'N', 'P_K_Ratio', 'NPK_Total', 'Temperature_Rainfall', 'rainfall', 'N_K_Ratio', 'Humidity_Rainfall', 'Temperature_Humidity', 'P', 'label']

AGRI YIELD TRAIN SELECTED


,NDVI_Temperature,OrganicMatter_Sand,Temperature_Rainfall,sand_pct,NDVI_Rainfall,soil_ph,organic_matter,Humidity_Rainfall,Temperature_Humidity,rainfall,yield
0,11.592,110.656,2777.04,41.6,50.692,6.25,2.66,7714.0,1764.0,110.2,4105.8
1,15.871,147.132,3397.47,40.2,74.517,6.70,3.66,9851.4,2098.2,126.3,4247.8
2,17.664,138.000,3019.44,40.0,70.016,7.21,3.45,7220.4,1821.6,109.4,4021.8
3,18.352,141.112,4596.88,56.9,96.286,5.73,2.48,12268.7,2338.4,155.3,4326.8
4,15.500,110.466,4315.00,30.6,107.012,6.88,3.61,13808.0,2000.0,172.6,3708.0



Columns:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall', 'yield']

AGRI YIELD TEST SELECTED


,NDVI_Temperature,OrganicMatter_Sand,Temperature_Rainfall,sand_pct,NDVI_Rainfall,soil_ph,organic_matter,Humidity_Rainfall,Temperature_Humidity,rainfall
0,16.740,101.480,4924.35,47.2,105.900,6.25,2.15,14473.0,2287.8,176.5
1,12.500,114.904,4187.50,54.2,83.750,7.40,2.12,12395.0,1850.0,167.5
2,12.556,199.056,4528.92,52.8,66.693,6.96,3.77,11632.5,2190.0,155.1
3,12.465,73.352,3423.72,34.6,55.620,6.70,2.12,10011.6,2243.7,123.6
4,15.936,75.210,4310.19,34.5,110.784,5.81,2.18,14540.4,2091.6,173.1



Columns:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall']


In [9]:
# ============================================================
# STAGE 6.4 — DATA TYPES
# ============================================================

print("========== CROP RECOMMENDATION ==========")
display(sf24.dtypes.to_frame("Data Type"))

print("\n========== AGRI YIELD TRAIN ==========")
display(agriyield_train.dtypes.to_frame("Data Type"))

print("\n========== AGRI YIELD TEST ==========")
display(agriyield_test.dtypes.to_frame("Data Type"))

========== CROP RECOMMENDATION ==========


,Data Type
humidity,float64
K,int64
temperature,float64
N,int64
P_K_Ratio,float64
NPK_Total,int64
Temperature_Rainfall,float64
rainfall,float64
N_K_Ratio,float64
Humidity_Rainfall,float64



========== AGRI YIELD TRAIN ==========


,Data Type
NDVI_Temperature,float64
OrganicMatter_Sand,float64
Temperature_Rainfall,float64
sand_pct,float64
NDVI_Rainfall,float64
soil_ph,float64
organic_matter,float64
Humidity_Rainfall,float64
Temperature_Humidity,float64
rainfall,float64



========== AGRI YIELD TEST ==========


,Data Type
NDVI_Temperature,float64
OrganicMatter_Sand,float64
Temperature_Rainfall,float64
sand_pct,float64
NDVI_Rainfall,float64
soil_ph,float64
organic_matter,float64
Humidity_Rainfall,float64
Temperature_Humidity,float64
rainfall,float64


In [10]:
# ============================================================
# STAGE 6.5 — MISSING VALUE CHECK
# ============================================================

print("========== CROP RECOMMENDATION ==========")
display(
    sf24.isnull()
    .sum()
    .to_frame("Missing Values")
)

print("\n========== AGRI YIELD TRAIN ==========")
display(
    agriyield_train.isnull()
    .sum()
    .to_frame("Missing Values")
)

print("\n========== AGRI YIELD TEST ==========")
display(
    agriyield_test.isnull()
    .sum()
    .to_frame("Missing Values")
)

========== CROP RECOMMENDATION ==========


,Missing Values
humidity,0
K,0
temperature,0
N,0
P_K_Ratio,0
NPK_Total,0
Temperature_Rainfall,0
rainfall,0
N_K_Ratio,0
Humidity_Rainfall,0



========== AGRI YIELD TRAIN ==========


,Missing Values
NDVI_Temperature,0
OrganicMatter_Sand,0
Temperature_Rainfall,0
sand_pct,0
NDVI_Rainfall,0
soil_ph,0
organic_matter,0
Humidity_Rainfall,0
Temperature_Humidity,0
rainfall,0



========== AGRI YIELD TEST ==========


,Missing Values
NDVI_Temperature,0
OrganicMatter_Sand,0
Temperature_Rainfall,0
sand_pct,0
NDVI_Rainfall,0
soil_ph,0
organic_matter,0
Humidity_Rainfall,0
Temperature_Humidity,0
rainfall,0


In [11]:
# ============================================================
# STAGE 6.6 — TARGET CANDIDATE CHECK
# ============================================================

print("Crop recommendation columns:")
print(sf24.columns.tolist())

print("\nAgriYield train columns:")
print(agriyield_train.columns.tolist())

Crop recommendation columns:
['humidity', 'K', 'temperature', 'N', 'P_K_Ratio', 'NPK_Total', 'Temperature_Rainfall', 'rainfall', 'N_K_Ratio', 'Humidity_Rainfall', 'Temperature_Humidity', 'P', 'label']

AgriYield train columns:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall', 'yield']


In [23]:
# ============================================================
# STAGE 6.8 — TARGET MISSING VALUES
# ============================================================

print("=" * 60)
print("TARGET MISSING VALUE AUDIT")
print("=" * 60)

# SF24 target
print("\nSF24 target:", SF24_TARGET)
print("Missing values:", sf24[SF24_TARGET].isnull().sum())

# AgriYield training target
print("\nAgriYield train target:", AGRIYIELD_TARGET)
print("Missing values:", agriyield_train[AGRIYIELD_TARGET].isnull().sum())

# AgriYield test
print("\nAgriYield test target:")
if AGRIYIELD_TARGET in agriyield_test.columns:
    print("Target is present.")
    print("Missing values:", agriyield_test[AGRIYIELD_TARGET].isnull().sum())
else:
    print("Target is not present in the test table.")
    print("This is expected because the test table contains features only.")

TARGET MISSING VALUE AUDIT

SF24 target: label
Missing values: 0

AgriYield train target: yield
Missing values: 0

AgriYield test target:
Target is not present in the test table.
This is expected because the test table contains features only.


In [13]:
# ============================================================
# STAGE 6.9 — POTENTIAL DATA LEAKAGE COLUMN AUDIT
# ============================================================

suspicious_keywords = [
    "target",
    "label",
    "class",
    "crop",
    "yield",
    "harvest",
    "harvested",
    "actual",
    "prediction",
    "predicted",
    "recommend",
    "recommended",
    "outcome",
    "result"
]


def leakage_keyword_audit(df, name, target):
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    suspicious_columns = []

    for col in df.columns:

        if col == target:
            continue

        col_lower = col.lower()

        matched_keywords = [
            keyword
            for keyword in suspicious_keywords
            if keyword in col_lower
        ]

        if matched_keywords:
            suspicious_columns.append({
                "Column": col,
                "Matched_Keywords": ", ".join(matched_keywords)
            })

    if suspicious_columns:
        report = pd.DataFrame(suspicious_columns)
        display(report)
    else:
        print("No suspicious column names found.")


leakage_keyword_audit(
    sf24,
    "SF24 Leakage Keyword Audit",
    SF24_TARGET
)

leakage_keyword_audit(
    agriyield_train,
    "AgriYield Leakage Keyword Audit",
    AGRIYIELD_TARGET
)


SF24 Leakage Keyword Audit


,Column,Matched_Keywords
0,label,label



AgriYield Leakage Keyword Audit
No suspicious column names found.


In [14]:
# ============================================================
# STAGE 6.10 — HIGH-CARDINALITY / ID-LIKE FEATURE AUDIT
# ============================================================

def high_cardinality_audit(df, name, target, threshold=0.95):

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    results = []

    for col in df.columns:

        if col == target:
            continue

        unique_ratio = df[col].nunique(dropna=False) / len(df)

        if unique_ratio >= threshold:

            results.append({
                "Column": col,
                "Unique_Count": df[col].nunique(dropna=False),
                "Unique_Ratio": round(unique_ratio, 4),
                "Data_Type": str(df[col].dtype)
            })

    if results:
        report = pd.DataFrame(results)
        display(report)
    else:
        print("No high-cardinality features found.")


high_cardinality_audit(
    sf24,
    "SF24 High Cardinality Audit",
    SF24_TARGET
)

high_cardinality_audit(
    agriyield_train,
    "AgriYield High Cardinality Audit",
    AGRIYIELD_TARGET
)


SF24 High Cardinality Audit


,Column,Unique_Count,Unique_Ratio,Data_Type
0,humidity,2200,1.0,float64
1,temperature,2200,1.0,float64
2,Temperature_Rainfall,2200,1.0,float64
3,rainfall,2200,1.0,float64
4,Humidity_Rainfall,2200,1.0,float64
5,Temperature_Humidity,2200,1.0,float64



AgriYield High Cardinality Audit
No high-cardinality features found.


In [15]:
# ============================================================
# STAGE 6.11 — CONSTANT FEATURE AUDIT
# ============================================================

def constant_feature_audit(df, name, target):

    feature_columns = [
        col for col in df.columns
        if col != target
    ]

    constant_columns = [
        col
        for col in feature_columns
        if df[col].nunique(dropna=False) <= 1
    ]

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    if constant_columns:
        print("Constant features:")
        print(constant_columns)
    else:
        print("No constant features found.")


constant_feature_audit(
    sf24,
    "SF24",
    SF24_TARGET
)

constant_feature_audit(
    agriyield_train,
    "AgriYield Train",
    AGRIYIELD_TARGET
)


SF24
No constant features found.

AgriYield Train
No constant features found.


In [16]:
# ============================================================
# STAGE 6.12 — TIME FEATURE AUDIT
# ============================================================

time_keywords = [
    "date",
    "year",
    "month",
    "season",
    "time",
    "day",
    "sowing",
    "planting",
    "harvest"
]


def time_feature_audit(df, name, target):

    results = []

    for col in df.columns:

        if col == target:
            continue

        col_lower = col.lower()

        matched = [
            keyword
            for keyword in time_keywords
            if keyword in col_lower
        ]

        if matched:
            results.append({
                "Column": col,
                "Matched_Keywords": ", ".join(matched),
                "Data_Type": str(df[col].dtype)
            })

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    if results:
        display(pd.DataFrame(results))
    else:
        print("No obvious time-related columns found.")


time_feature_audit(
    sf24,
    "SF24 Time Feature Audit",
    SF24_TARGET
)

time_feature_audit(
    agriyield_train,
    "AgriYield Time Feature Audit",
    AGRIYIELD_TARGET
)


SF24 Time Feature Audit
No obvious time-related columns found.

AgriYield Time Feature Audit
No obvious time-related columns found.


In [17]:
# ============================================================
# STAGE 6.13 — GROUP / LOCATION FEATURE AUDIT
# ============================================================

group_keywords = [
    "farm",
    "field",
    "plot",
    "location",
    "region",
    "district",
    "state",
    "country",
    "latitude",
    "longitude",
    "lat",
    "lon",
    "site",
    "station"
]


def group_feature_audit(df, name, target):

    results = []

    for col in df.columns:

        if col == target:
            continue

        col_lower = col.lower()

        matched = [
            keyword
            for keyword in group_keywords
            if keyword in col_lower
        ]

        if matched:
            results.append({
                "Column": col,
                "Matched_Keywords": ", ".join(matched),
                "Unique_Count": df[col].nunique(dropna=False)
            })

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    if results:
        display(pd.DataFrame(results))
    else:
        print("No obvious group/location columns found.")


group_feature_audit(
    sf24,
    "SF24 Group / Location Audit",
    SF24_TARGET
)

group_feature_audit(
    agriyield_train,
    "AgriYield Group / Location Audit",
    AGRIYIELD_TARGET
)


SF24 Group / Location Audit
No obvious group/location columns found.

AgriYield Group / Location Audit
No obvious group/location columns found.


In [20]:
# ============================================================
# CHECK ACTUAL TARGET COLUMNS
# ============================================================

print("SF24 columns:")
print(sf24.columns.tolist())

print("\nAgriYield Train columns:")
print(agriyield_train.columns.tolist())

print("\nAgriYield Test columns:")
print(agriyield_test.columns.tolist())

SF24 columns:
['humidity', 'K', 'temperature', 'N', 'P_K_Ratio', 'NPK_Total', 'Temperature_Rainfall', 'rainfall', 'N_K_Ratio', 'Humidity_Rainfall', 'Temperature_Humidity', 'P', 'label']

AgriYield Train columns:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall', 'yield']

AgriYield Test columns:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall']


In [21]:
# ============================================================
# STAGE 6.14 — SEPARATE FEATURES AND TARGETS
# ============================================================

# -------------------------
# SF24
# -------------------------

SF24_TARGET = "label"

X_sf24 = sf24.drop(columns=[SF24_TARGET])
y_sf24 = sf24[SF24_TARGET]


# -------------------------
# AgriYield
# -------------------------

AGRIYIELD_TARGET = "yield"

X_agri_train = agriyield_train.drop(columns=[AGRIYIELD_TARGET])
y_agri_train = agriyield_train[AGRIYIELD_TARGET]

# Test table contains features only
X_agri_test = agriyield_test.copy()


print("=" * 60)
print("FEATURE / TARGET SHAPES")
print("=" * 60)

print("SF24 X:", X_sf24.shape)
print("SF24 y:", y_sf24.shape)

print("\nAgriYield Train X:", X_agri_train.shape)
print("AgriYield Train y:", y_agri_train.shape)

print("\nAgriYield Test X:", X_agri_test.shape)

FEATURE / TARGET SHAPES
SF24 X: (2200, 12)
SF24 y: (2200,)

AgriYield Train X: (10000, 10)
AgriYield Train y: (10000,)

AgriYield Test X: (2000, 10)


In [24]:
# ============================================================
# STAGE 6.15 — FEATURE TYPES
# ============================================================

# SF24
sf24_numeric_features = X_sf24.select_dtypes(
    include=np.number
).columns.tolist()

sf24_categorical_features = X_sf24.select_dtypes(
    exclude=np.number
).columns.tolist()


# AgriYield
agri_numeric_features = X_agri_train.select_dtypes(
    include=np.number
).columns.tolist()

agri_categorical_features = X_agri_train.select_dtypes(
    exclude=np.number
).columns.tolist()


print("=" * 60)
print("SF24 FEATURE TYPES")
print("=" * 60)

print("Numerical features:")
print(sf24_numeric_features)

print("\nCategorical features:")
print(sf24_categorical_features)


print("\n" + "=" * 60)
print("AGRIYIELD FEATURE TYPES")
print("=" * 60)

print("Numerical features:")
print(agri_numeric_features)

print("\nCategorical features:")
print(agri_categorical_features)

SF24 FEATURE TYPES
Numerical features:
['humidity', 'K', 'temperature', 'N', 'P_K_Ratio', 'NPK_Total', 'Temperature_Rainfall', 'rainfall', 'N_K_Ratio', 'Humidity_Rainfall', 'Temperature_Humidity', 'P']

Categorical features:
[]

AGRIYIELD FEATURE TYPES
Numerical features:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall']

Categorical features:
[]


In [25]:
# ============================================================
# STAGE 6.16 — SF24 TRAIN/TEST SPLIT
# ============================================================

TEST_SIZE = 0.20
RANDOM_STATE = 42

X_sf24_train, X_sf24_test, y_sf24_train, y_sf24_test = train_test_split(
    X_sf24,
    y_sf24,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_sf24
)

print("=" * 60)
print("SF24 TRAIN / TEST SPLIT")
print("=" * 60)

print("Training features :", X_sf24_train.shape)
print("Testing features  :", X_sf24_test.shape)

print("Training target   :", y_sf24_train.shape)
print("Testing target    :", y_sf24_test.shape)

SF24 TRAIN / TEST SPLIT
Training features : (1760, 12)
Testing features  : (440, 12)
Training target   : (1760,)
Testing target    : (440,)


In [26]:
# ============================================================
# STAGE 6.17 — VERIFY SF24 STRATIFICATION
# ============================================================

train_distribution = (
    y_sf24_train
    .value_counts(normalize=True)
    .sort_index()
    .rename("Train_Percentage")
    * 100
)

test_distribution = (
    y_sf24_test
    .value_counts(normalize=True)
    .sort_index()
    .rename("Test_Percentage")
    * 100
)

class_distribution = pd.concat(
    [
        train_distribution,
        test_distribution
    ],
    axis=1
)

print("=" * 60)
print("SF24 CLASS DISTRIBUTION")
print("=" * 60)

display(class_distribution.round(2))

SF24 CLASS DISTRIBUTION


,Train_Percentage,Test_Percentage
label,,
apple,4.55,4.55
banana,4.55,4.55
blackgram,4.55,4.55
chickpea,4.55,4.55
coconut,4.55,4.55
coffee,4.55,4.55
cotton,4.55,4.55
grapes,4.55,4.55
jute,4.55,4.55


In [27]:
# ============================================================
# STAGE 6.18 — AGRIYIELD PREDEFINED TRAIN / TEST STRUCTURE
# ============================================================

print("=" * 60)
print("AGRIYIELD TRAIN / TEST STRUCTURE")
print("=" * 60)

print("AgriYield Train rows :", len(agriyield_train))
print("AgriYield Test rows  :", len(agriyield_test))

print("\nTrain features :", X_agri_train.shape)
print("Train target   :", y_agri_train.shape)

print("\nTest features  :", X_agri_test.shape)

print(
    "\nTest target available:",
    AGRIYIELD_TARGET in agriyield_test.columns
)

AGRIYIELD TRAIN / TEST STRUCTURE
AgriYield Train rows : 10000
AgriYield Test rows  : 2000

Train features : (10000, 10)
Train target   : (10000,)

Test features  : (2000, 10)

Test target available: False


In [28]:
# ============================================================
# STAGE 6.19 — AGRIYIELD FEATURE CONSISTENCY
# ============================================================

train_columns = set(X_agri_train.columns)
test_columns = set(X_agri_test.columns)

train_only = train_columns - test_columns
test_only = test_columns - train_columns

print("=" * 60)
print("AGRIYIELD FEATURE CONSISTENCY")
print("=" * 60)

print("Features only in train:")
print(train_only)

print("\nFeatures only in test:")
print(test_only)

if not train_only and not test_only:
    print("\nTrain and test feature columns are consistent.")
else:
    print("\nWARNING: Train/test feature columns are different.")

AGRIYIELD FEATURE CONSISTENCY
Features only in train:
set()

Features only in test:
set()

Train and test feature columns are consistent.


In [29]:
# ============================================================
# STAGE 6.20 — AGRIYIELD TRAIN/TEST OVERLAP
# ============================================================

common_columns = X_agri_train.columns.tolist()

train_feature_keys = (
    X_agri_train[common_columns]
    .astype(str)
    .agg("|".join, axis=1)
)

test_feature_keys = (
    X_agri_test[common_columns]
    .astype(str)
    .agg("|".join, axis=1)
)

overlap_count = len(
    set(train_feature_keys) &
    set(test_feature_keys)
)

print("=" * 60)
print("AGRIYIELD TRAIN/TEST FEATURE OVERLAP")
print("=" * 60)

print("Exact overlapping feature rows:", overlap_count)

if overlap_count == 0:
    print("No exact feature-row overlap detected.")
else:
    print("WARNING: Exact feature-row overlap detected.")

AGRIYIELD TRAIN/TEST FEATURE OVERLAP
Exact overlapping feature rows: 0
No exact feature-row overlap detected.


In [30]:
# ============================================================
# STAGE 6.21 — SF24 PREPROCESSING PIPELINE
# ============================================================

sf24_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

sf24_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

sf24_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            sf24_numeric_pipeline,
            sf24_numeric_features
        ),
        (
            "categorical",
            sf24_categorical_pipeline,
            sf24_categorical_features
        )
    ],
    remainder="drop"
)

print("SF24 preprocessing pipeline created.")

SF24 preprocessing pipeline created.


In [31]:
# ============================================================
# STAGE 6.21 — SF24 PREPROCESSING PIPELINE
# ============================================================

sf24_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

sf24_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

sf24_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            sf24_numeric_pipeline,
            sf24_numeric_features
        ),
        (
            "categorical",
            sf24_categorical_pipeline,
            sf24_categorical_features
        )
    ],
    remainder="drop"
)

print("SF24 preprocessing pipeline created.")

SF24 preprocessing pipeline created.


In [32]:
# ============================================================
# STAGE 6.22 — AGRIYIELD PREPROCESSING PIPELINE
# ============================================================

agri_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

agri_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

agri_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            agri_numeric_pipeline,
            agri_numeric_features
        ),
        (
            "categorical",
            agri_categorical_pipeline,
            agri_categorical_features
        )
    ],
    remainder="drop"
)

print("AgriYield preprocessing pipeline created.")

AgriYield preprocessing pipeline created.


In [33]:
# ============================================================
# STAGE 6.23 — SF24 CROSS-VALIDATION STRATEGY
# ============================================================

N_SPLITS = 5

sf24_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("=" * 60)
print("SF24 CROSS-VALIDATION STRATEGY")
print("=" * 60)

print("Method          : StratifiedKFold")
print("Number of folds :", N_SPLITS)
print("Shuffle         : True")
print("Random state    :", RANDOM_STATE)

SF24 CROSS-VALIDATION STRATEGY
Method          : StratifiedKFold
Number of folds : 5
Shuffle         : True
Random state    : 42


In [34]:
# ============================================================
# STAGE 6.23 — SF24 CROSS-VALIDATION STRATEGY
# ============================================================

N_SPLITS = 5

sf24_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("=" * 60)
print("SF24 CROSS-VALIDATION STRATEGY")
print("=" * 60)

print("Method          : StratifiedKFold")
print("Number of folds :", N_SPLITS)
print("Shuffle         : True")
print("Random state    :", RANDOM_STATE)

SF24 CROSS-VALIDATION STRATEGY
Method          : StratifiedKFold
Number of folds : 5
Shuffle         : True
Random state    : 42


In [35]:
# ============================================================
# STAGE 6.25 — SF24 BASELINE PIPELINE
# ============================================================

sf24_baseline = Pipeline([
    (
        "preprocessor",
        sf24_preprocessor
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    )
])

print("SF24 baseline pipeline created.")

SF24 baseline pipeline created.


In [36]:
# ============================================================
# STAGE 6.26 — SF24 CROSS-VALIDATION
# ============================================================

sf24_scoring = {
    "accuracy": "accuracy",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted"
}

sf24_cv_results = cross_validate(
    sf24_baseline,
    X_sf24_train,
    y_sf24_train,
    cv=sf24_cv,
    scoring=sf24_scoring,
    return_train_score=False
)

sf24_cv_results_df = pd.DataFrame({
    "Accuracy": sf24_cv_results["test_accuracy"],
    "Precision": sf24_cv_results["test_precision_weighted"],
    "Recall": sf24_cv_results["test_recall_weighted"],
    "F1": sf24_cv_results["test_f1_weighted"]
})

print("=" * 60)
print("SF24 CROSS-VALIDATION RESULTS")
print("=" * 60)

display(sf24_cv_results_df.round(4))

SF24 CROSS-VALIDATION RESULTS


,Accuracy,Precision,Recall,F1
0,0.9830,0.9838,0.9830,0.9829
1,0.9659,0.9684,0.9659,0.9660
2,0.9659,0.9699,0.9659,0.9651
3,0.9716,0.9749,0.9716,0.9713
4,0.9659,0.9692,0.9659,0.9663


In [37]:
# ============================================================
# STAGE 6.26 — SF24 CROSS-VALIDATION
# ============================================================

sf24_scoring = {
    "accuracy": "accuracy",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted"
}

sf24_cv_results = cross_validate(
    sf24_baseline,
    X_sf24_train,
    y_sf24_train,
    cv=sf24_cv,
    scoring=sf24_scoring,
    return_train_score=False
)

sf24_cv_results_df = pd.DataFrame({
    "Accuracy": sf24_cv_results["test_accuracy"],
    "Precision": sf24_cv_results["test_precision_weighted"],
    "Recall": sf24_cv_results["test_recall_weighted"],
    "F1": sf24_cv_results["test_f1_weighted"]
})

print("=" * 60)
print("SF24 CROSS-VALIDATION RESULTS")
print("=" * 60)

display(sf24_cv_results_df.round(4))

SF24 CROSS-VALIDATION RESULTS


,Accuracy,Precision,Recall,F1
0,0.9830,0.9838,0.9830,0.9829
1,0.9659,0.9684,0.9659,0.9660
2,0.9659,0.9699,0.9659,0.9651
3,0.9716,0.9749,0.9716,0.9713
4,0.9659,0.9692,0.9659,0.9663


In [38]:
# ============================================================
# STAGE 6.28 — SF24 HOLDOUT TEST EVALUATION
# ============================================================

sf24_baseline.fit(
    X_sf24_train,
    y_sf24_train
)

sf24_test_predictions = sf24_baseline.predict(
    X_sf24_test
)

sf24_test_accuracy = accuracy_score(
    y_sf24_test,
    sf24_test_predictions
)

print("=" * 60)
print("SF24 HOLDOUT TEST RESULT")
print("=" * 60)

print("Accuracy:", round(sf24_test_accuracy, 4))

SF24 HOLDOUT TEST RESULT
Accuracy: 0.9727


In [39]:
# ============================================================
# STAGE 6.29 — AGRIYIELD BASELINE PIPELINE
# ============================================================

agri_baseline = Pipeline([
    (
        "preprocessor",
        agri_preprocessor
    ),
    (
        "model",
        LinearRegression()
    )
])

print("AgriYield baseline pipeline created.")

AgriYield baseline pipeline created.


In [42]:
# ============================================================
# STAGE 6.30 — AGRIYIELD CROSS-VALIDATION
# ============================================================

agri_scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

agri_cv_results = cross_validate(
    agri_baseline,
    X_agri_train,
    y_agri_train,
    cv=agri_cv,
    scoring=agri_scoring,
    return_train_score=False
)

agri_cv_results_df = pd.DataFrame({
    "MAE": -agri_cv_results["test_MAE"],
    "RMSE": np.sqrt(-agri_cv_results["test_MSE"]),
    "R2": agri_cv_results["test_R2"]
})

print("=" * 60)
print("AGRIYIELD CROSS-VALIDATION RESULTS")
print("=" * 60)

display(agri_cv_results_df.round(4))

AGRIYIELD CROSS-VALIDATION RESULTS


,MAE,RMSE,R2
0,376.7553,435.4530,-0.0008
1,380.4208,436.4718,-0.0007
2,384.1085,443.5025,-0.0015
3,383.2886,439.7396,-0.0027
4,369.1782,428.1390,0.0001


In [41]:
# ============================================================
# STAGE 6.24 — AGRIYIELD CROSS-VALIDATION STRATEGY
# ============================================================

N_SPLITS = 5
RANDOM_STATE = 42

agri_cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("=" * 60)
print("AGRIYIELD CROSS-VALIDATION STRATEGY")
print("=" * 60)

print("Method          : KFold")
print("Number of folds :", N_SPLITS)
print("Shuffle         : True")
print("Random state    :", RANDOM_STATE)

AGRIYIELD CROSS-VALIDATION STRATEGY
Method          : KFold
Number of folds : 5
Shuffle         : True
Random state    : 42


In [43]:
# ============================================================
# STAGE 6.31 — AGRIYIELD CV SUMMARY
# ============================================================

agri_cv_summary = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R2"
    ],
    "Mean": [
        agri_cv_results_df["MAE"].mean(),
        agri_cv_results_df["RMSE"].mean(),
        agri_cv_results_df["R2"].mean()
    ],
    "Std": [
        agri_cv_results_df["MAE"].std(),
        agri_cv_results_df["RMSE"].std(),
        agri_cv_results_df["R2"].std()
    ]
})

print("=" * 60)
print("AGRIYIELD CROSS-VALIDATION SUMMARY")
print("=" * 60)

display(agri_cv_summary.round(4))

AGRIYIELD CROSS-VALIDATION SUMMARY


,Metric,Mean,Std
0,MAE,378.7503,6.0766
1,RMSE,436.6612,5.7097
2,R2,-0.0011,0.0010


In [44]:
# ============================================================
# STAGE 6.32 — TRAIN AGRIYIELD BASELINE
# ============================================================

agri_baseline.fit(
    X_agri_train,
    y_agri_train
)

print("AgriYield baseline trained successfully.")

AgriYield baseline trained successfully.


In [45]:
# ============================================================
# STAGE 6.33 — AGRIYIELD TEST PREDICTIONS
# ============================================================

agri_test_predictions = agri_baseline.predict(
    X_agri_test
)

print("=" * 60)
print("AGRIYIELD TEST PREDICTIONS")
print("=" * 60)

print("Number of predictions:", len(agri_test_predictions))

display(
    pd.DataFrame({
        "Predicted_Yield": agri_test_predictions
    }).head(10)
)

AGRIYIELD TEST PREDICTIONS
Number of predictions: 2000


,Predicted_Yield
0,4259.350699
1,4288.650345
2,4238.443570
3,4229.374547
4,4255.917044
5,4233.837025
6,4248.369040
7,4276.245619
8,4258.729995
9,4257.553708


In [46]:
# ============================================================
# STAGE 6.34 — FEATURE SELECTION LEAKAGE CHECK
# ============================================================

print("=" * 60)
print("FEATURE SELECTION LEAKAGE CHECK")
print("=" * 60)

print("""
Stage 5 created the selected-feature tables.

For strict leakage prevention, feature selection should ideally
be fitted only on training data.

Therefore, if feature selection is performed again during
model development or hyperparameter tuning, it should be
placed inside the ML pipeline.

Stage 6 starts from the selected tables created in Stage 5.
""")

FEATURE SELECTION LEAKAGE CHECK

Stage 5 created the selected-feature tables.

For strict leakage prevention, feature selection should ideally
be fitted only on training data.

Therefore, if feature selection is performed again during
model development or hyperparameter tuning, it should be
placed inside the ML pipeline.

Stage 6 starts from the selected tables created in Stage 5.



In [47]:
# ============================================================
# STAGE 6.35 — AGRIYIELD TARGET CORRELATION SCREEN
# ============================================================

numeric_agri = agriyield_train.select_dtypes(
    include=np.number
)

correlation_with_target = (
    numeric_agri
    .corr()[AGRIYIELD_TARGET]
    .drop(AGRIYIELD_TARGET)
    .sort_values(
        key=lambda x: abs(x),
        ascending=False
    )
)

print("=" * 60)
print("NUMERIC FEATURE CORRELATION WITH YIELD")
print("=" * 60)

display(
    correlation_with_target.to_frame(
        "Correlation_With_Yield"
    ).round(4)
)

NUMERIC FEATURE CORRELATION WITH YIELD


,Correlation_With_Yield
rainfall,0.0170
Humidity_Rainfall,0.0155
Temperature_Rainfall,0.0110
NDVI_Rainfall,0.0083
Temperature_Humidity,-0.0081
soil_ph,0.0071
NDVI_Temperature,-0.0065
OrganicMatter_Sand,-0.0045
sand_pct,-0.0039
organic_matter,0.0022


In [48]:
# ============================================================
# STAGE 6.36 — FINAL STAGE 6 RESULTS
# ============================================================

sf24_final_results = pd.DataFrame({
    "Metric": [
        "CV Accuracy",
        "CV Precision",
        "CV Recall",
        "CV F1",
        "Holdout Accuracy"
    ],
    "Value": [
        sf24_cv_results_df["Accuracy"].mean(),
        sf24_cv_results_df["Precision"].mean(),
        sf24_cv_results_df["Recall"].mean(),
        sf24_cv_results_df["F1"].mean(),
        sf24_test_accuracy
    ]
})


agri_final_results = pd.DataFrame({
    "Metric": [
        "CV MAE",
        "CV RMSE",
        "CV R2",
        "Test Predictions Generated"
    ],
    "Value": [
        agri_cv_results_df["MAE"].mean(),
        agri_cv_results_df["RMSE"].mean(),
        agri_cv_results_df["R2"].mean(),
        len(agri_test_predictions)
    ]
})


print("=" * 60)
print("SF24 FINAL RESULTS")
print("=" * 60)

display(sf24_final_results.round(4))


print("\n" + "=" * 60)
print("AGRIYIELD FINAL RESULTS")
print("=" * 60)

display(agri_final_results.round(4))

SF24 FINAL RESULTS


,Metric,Value
0,CV Accuracy,0.9705
1,CV Precision,0.9732
2,CV Recall,0.9705
3,CV F1,0.9703
4,Holdout Accuracy,0.9727



AGRIYIELD FINAL RESULTS


,Metric,Value
0,CV MAE,378.7503
1,CV RMSE,436.6612
2,CV R2,-0.0011
3,Test Predictions Generated,2000.0000


In [50]:
# ============================================================
# STAGE 6.27 — SF24 CV SUMMARY
# ============================================================

sf24_cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ],
    "Mean": [
        sf24_cv_results_df["Accuracy"].mean(),
        sf24_cv_results_df["Precision"].mean(),
        sf24_cv_results_df["Recall"].mean(),
        sf24_cv_results_df["F1"].mean()
    ],
    "Std": [
        sf24_cv_results_df["Accuracy"].std(),
        sf24_cv_results_df["Precision"].std(),
        sf24_cv_results_df["Recall"].std(),
        sf24_cv_results_df["F1"].std()
    ]
})

print("=" * 60)
print("SF24 CROSS-VALIDATION SUMMARY")
print("=" * 60)

display(sf24_cv_summary.round(4))

SF24 CROSS-VALIDATION SUMMARY


,Metric,Mean,Std
0,Accuracy,0.9705,0.0074
1,Precision,0.9732,0.0064
2,Recall,0.9705,0.0074
3,F1,0.9703,0.0074


In [51]:
# ============================================================
# STAGE 6.37 — SAVE STAGE 6 OUTPUTS
# ============================================================

stage6_output_dir = Path("stage6_outputs")

stage6_output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# -------------------------
# SF24 outputs
# -------------------------

sf24_cv_results_df.to_csv(
    stage6_output_dir / "sf24_cv_results.csv",
    index=False
)

sf24_cv_summary.to_csv(
    stage6_output_dir / "sf24_cv_summary.csv",
    index=False
)

sf24_final_results.to_csv(
    stage6_output_dir / "sf24_final_results.csv",
    index=False
)

class_distribution.to_csv(
    stage6_output_dir / "sf24_class_distribution.csv"
)


# -------------------------
# AgriYield outputs
# -------------------------

agri_cv_results_df.to_csv(
    stage6_output_dir / "agriyield_cv_results.csv",
    index=False
)

agri_cv_summary.to_csv(
    stage6_output_dir / "agriyield_cv_summary.csv",
    index=False
)

agri_final_results.to_csv(
    stage6_output_dir / "agriyield_final_results.csv",
    index=False
)

pd.DataFrame({
    "Predicted_Yield": agri_test_predictions
}).to_csv(
    stage6_output_dir / "agriyield_test_predictions.csv",
    index=False
)


print("=" * 60)
print("STAGE 6 OUTPUTS SAVED")
print("=" * 60)

print("Output directory:")
print(stage6_output_dir.resolve())

STAGE 6 OUTPUTS SAVED
Output directory:
C:\Users\ADMIN\smart agricultural analytics\stage6_outputs


In [52]:
# ============================================================
# STAGE 6.38 — FINAL STAGE 6 REPORT
# ============================================================

print("\n")
print("=" * 70)
print("STAGE 6 — DATA LEAKAGE & VALIDATION STRATEGY COMPLETED")
print("=" * 70)


# ============================================================
# 1. DATASETS
# ============================================================

print("\n1. DATASETS")
print("-" * 70)

print("SF24")
print("  Table  :", "crop_recommendation_selected")
print("  Target :", SF24_TARGET)
print("  Task   :", "Multiclass Classification")

print("\nAgriYield")
print("  Train table :", "agri_yield_train_selected")
print("  Test table  :", "agri_yield_test_selected")
print("  Target      :", AGRIYIELD_TARGET)
print("  Task        :", "Regression")


# ============================================================
# 2. VALIDATION STRATEGY
# ============================================================

print("\n2. VALIDATION STRATEGY")
print("-" * 70)

print("SF24")
print("  Train/Test split : 80/20")
print("  Stratification   : Yes")
print("  Cross-validation : StratifiedKFold")
print("  Folds            :", N_SPLITS)

print("\nAgriYield")
print("  Training data    : agri_yield_train_selected")
print("  Cross-validation : KFold")
print("  Folds            :", N_SPLITS)
print("  Test target      : Not available")


# ============================================================
# 3. LEAKAGE CONTROLS
# ============================================================

print("\n3. LEAKAGE CONTROLS")
print("-" * 70)

print("✓ Missing values checked")
print("✓ Target leakage audit performed")
print("✓ Duplicate rows checked")
print("✓ Constant features checked")
print("✓ Time-related features checked")
print("✓ Group/location features checked")
print("✓ Train/test feature consistency checked")
print("✓ Train/test feature-row overlap checked")
print("✓ Preprocessing placed inside pipelines")
print("✓ SF24 test set kept outside CV")
print("✓ AgriYield test features kept outside CV")


# ============================================================
# 4. BASELINE MODELS
# ============================================================

print("\n4. BASELINE MODELS")
print("-" * 70)

print("SF24      : Logistic Regression")
print("AgriYield : Linear Regression")


# ============================================================
# 5. RESULTS
# ============================================================

print("\n5. RESULTS")
print("-" * 70)

print("\nSF24 Results:")
display(sf24_final_results.round(4))

print("\nAgriYield Results:")
display(agri_final_results.round(4))


# ============================================================
# 6. OUTPUT DIRECTORY
# ============================================================

print("\n6. OUTPUT DIRECTORY")
print("-" * 70)

print(stage6_output_dir.resolve())


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("STAGE 6 COMPLETE")
print("=" * 70)

print("\nNext stage:")
print("STAGE 7 — BASELINE MODEL DEVELOPMENT & COMPARISON")
print("=" * 70)



STAGE 6 — DATA LEAKAGE & VALIDATION STRATEGY COMPLETED

1. DATASETS
----------------------------------------------------------------------
SF24
  Table  : crop_recommendation_selected
  Target : label
  Task   : Multiclass Classification

AgriYield
  Train table : agri_yield_train_selected
  Test table  : agri_yield_test_selected
  Target      : yield
  Task        : Regression

2. VALIDATION STRATEGY
----------------------------------------------------------------------
SF24
  Train/Test split : 80/20
  Stratification   : Yes
  Cross-validation : StratifiedKFold
  Folds            : 5

AgriYield
  Training data    : agri_yield_train_selected
  Cross-validation : KFold
  Folds            : 5
  Test target      : Not available

3. LEAKAGE CONTROLS
----------------------------------------------------------------------
✓ Missing values checked
✓ Target leakage audit performed
✓ Duplicate rows checked
✓ Constant features checked
✓ Time-related features checked
✓ Group/location features c

,Metric,Value
0,CV Accuracy,0.9705
1,CV Precision,0.9732
2,CV Recall,0.9705
3,CV F1,0.9703
4,Holdout Accuracy,0.9727



AgriYield Results:


,Metric,Value
0,CV MAE,378.7503
1,CV RMSE,436.6612
2,CV R2,-0.0011
3,Test Predictions Generated,2000.0000



6. OUTPUT DIRECTORY
----------------------------------------------------------------------
C:\Users\ADMIN\smart agricultural analytics\stage6_outputs

STAGE 6 COMPLETE

Next stage:
STAGE 7 — BASELINE MODEL DEVELOPMENT & COMPARISON
